# Three maps, three kinds of colour (Python)

**Fullstack Remote Sensing** by Rifky Nauval Hendrawan · [Chapter: Data Visualisation That Decides Things](https://rifkynhsp-byte.github.io/Fullstack-Remote-Sensing/en/29-dataviz.html) · [Bab (Bahasa Indonesia)](https://rifkynhsp-byte.github.io/Fullstack-Remote-Sensing/id/29-dataviz.html) · [Open the JavaScript in the Code Editor](https://code.earthengine.google.com/?scriptPath=users/rifkynauvalhsp/Fullstack-Remote-Sensing:ch29_rainfall_maps)

Run the cells from top to bottom. The first cell installs what Colab lacks, fetches the book's data and helpers, and signs you in to Earth Engine with your own Google Cloud project ([how to get one](https://developers.google.com/earth-engine/guides/access)).

*Jalankan sel dari atas ke bawah. Sel pertama memasang paket, mengambil data buku, dan masuk ke Earth Engine dengan proyek Google Cloud Anda sendiri.*

In [ ]:
# @title Setup: packages, book data, Earth Engine sign-in
!pip -q install earthengine-api geemap tabulate
!test -d /content/book || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/rifkynhsp-byte/Fullstack-Remote-Sensing.git /content/book
!git -C /content/book sparse-checkout set tools scripts data   # only what the code needs, not the website
import sys, os; sys.path[:0] = ['/content/book/tools', '/content/book/scripts/py']; os.chdir('/content/book')
__file__ = '/content/book/scripts/py/ch29_rainfall_maps.py'   # the script finds the book's data relative to itself
PROJECT = "your-cloud-project"  # @param {type:"string"}
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT)
_init = ee.Initialize; ee.Initialize = lambda *a, **k: None   # the script must not sign in again

CHAPTER 29 | One dataset (CHIRPS pentads), three questions, three colour jobs.

    How much rain falls in a year?          sequential: one hue, light to dark
    Was 2023 wetter or drier than normal?   diverging: two hues, neutral at zero
    In which season is the wettest month?   categorical: distinct hues, no order

In [ ]:
import ee

indonesia = ee.Geometry.Rectangle([94.8, -11.2, 141.2, 6.2], None, False)
pentads = (ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD").select("precipitation")
           .filterBounds(indonesia))
BASE = ("1991-01-01", "2021-01-01")          # the 1991-2020 normal period

# 1. Sequential: mean annual rainfall, 1991-2020 (mm per year)
annual_mean = pentads.filterDate(*BASE).sum().divide(30).rename("rain")

# 2. Diverging: 2023 as a percentage of that normal
rain_2023 = pentads.filterDate("2023-01-01", "2024-01-01").sum()
anomaly_pct = rain_2023.divide(annual_mean).subtract(1).multiply(100).rename("anom")

# 3. Categorical: the season that holds the wettest calendar month
normal = pentads.filterDate(*BASE)
monthly_clim = ee.ImageCollection([
    normal.filter(ee.Filter.calendarRange(m, m, "month")).sum().divide(30)
    .rename("rain").set("month", m) for m in range(1, 13)])
# Stack the 12 monthly means as an array and take the index of the largest.
wettest = monthly_clim.toArray().arrayProject([0]).arrayArgmax().arrayGet([0]).add(1)
# Dec-Feb = 0, Mar-May = 1, Jun-Aug = 2, Sep-Nov = 3
season = wettest.mod(12).divide(3).floor().rename("season")

land = annual_mean.mask()
SEQ = ["f7fbff", "deebf7", "c6dbef", "9ecae1", "6baed6", "4292c6", "2171b5",
       "08519c", "08306b"]                                   # one hue: blues
DIV = ["8c510a", "bf812d", "dfc27d", "f6e8c3", "f5f5f5", "c7eae5", "80cdc1",
       "35978f", "01665e"]                                   # brown to teal
CAT = ["0072B2", "E69F00", "D55E00", "009E73"]               # Okabe-Ito
SRC = "CHIRPS v2.0 pentads (Funk et al. 2015), Google Earth Engine"

In [ ]:
def products():
    return [
        {"kind": "map", "name": "ch29-map-sequential", "image": annual_mean,
         "region": indonesia, "vis": {"min": 1000, "max": 4500, "palette": SEQ},
         "legend": "Mean annual rainfall, 1991-2020 (mm)", "width": 1600,
         "title": "How much? Mean annual rainfall", "source": SRC,
         "caption": "Sequential: one hue from light to dark, because the question has one "
                    "direction (more rain). The stretch runs from 1,000 to 4,500 mm; "
                    "values outside it are drawn at the ends."},
        {"kind": "map", "name": "ch29-map-diverging", "image": anomaly_pct,
         "region": indonesia, "vis": {"min": -50, "max": 50, "palette": DIV},
         "legend": "2023 rainfall compared with 1991-2020 (%)", "width": 1600,
         "title": "Wetter or drier? 2023 against normal", "source": SRC,
         "caption": "Diverging: brown for drier, teal for wetter, near-white for normal. "
                    "The colour scale is symmetric around zero, so equal colours mean "
                    "equal departures in both directions."},
        {"kind": "map", "name": "ch29-map-categorical", "image": season.updateMask(land),
         "region": indonesia, "vis": {"min": 0, "max": 3, "palette": CAT}, "width": 1600,
         "classes": [("Dec-Feb", "#0072B2"), ("Mar-May", "#E69F00"),
                     ("Jun-Aug", "#D55E00"), ("Sep-Nov", "#009E73")],
         "title": "Which season? The wettest month, 1991-2020", "source": SRC,
         "caption": "Categorical: four seasons, four unrelated hues from the Okabe-Ito "
                    "set, which stays distinct for the common forms of colour blindness."},
    ]

### The chapter's maps, charts and tables
The same products the book shows, drawn here. Maps also open as an interactive layer.

In [ ]:
from colab_render import show_all
show_all(products())